# ML_U2_C06 — Support Vector Machines (SVM)

📝 **Modalidad: Clase interactiva — sigue junto al profesor.**

**Versión:** 2025-1 | **Modificado:** 2026-05-03

---

## 🎯 Objetivos de la Clase

1. Explicar la intuición del clasificador de margen máximo.
2. Distinguir hard margin vs soft margin (parámetro **C**).
3. Aplicar el truco del kernel para datos no linealmente separables.
4. Entrenar SVMs con sklearn y comparar kernels.
5. **(Doctorado)** Derivar el problema dual y comprender el rol de los vectores de soporte.

---

## ⚙️ Setup (NO MODIFICAR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_moons, make_circles, load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ Imports completos")

## Sección 1 — La Intuición: Margen Máximo

Dado un conjunto linealmente separable, existen **infinitos** hiperplanos que lo separan.
SVM elige el hiperplano que **maximiza el margen** — la distancia mínima a los puntos de cada clase.

> 🔑 Mayor margen → mayor tolerancia a nuevos puntos → mejor generalización.

In [ ]:
# Dataset sintético linealmente separable
X_sep, y_sep = make_blobs(n_samples=50, centers=2, cluster_std=0.7,
                           center_box=(-3, 3), random_state=RANDOM_STATE)
y_sep = 2 * y_sep - 1  # → {-1, +1}

# Hard margin SVM
svm_hard = SVC(kernel='linear', C=1e6)
svm_hard.fit(X_sep, y_sep)

def plot_margin(ax, model, X, y, title):
    """Visualiza hiperplano, márgenes y vectores de soporte."""
    ax.set_xlim(X[:,0].min()-0.5, X[:,0].max()+0.5)
    ax.set_ylim(X[:,1].min()-0.5, X[:,1].max()+0.5)
    xx = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1], 300)
    yy = np.linspace(ax.get_ylim()[0], ax.get_ylim()[1], 300)
    XX, YY = np.meshgrid(xx, yy)
    Z = model.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contour(XX, YY, Z, levels=[-1, 0, 1],
               linestyles=['--', '-', '--'], colors=['#E74C3C','#2C3E50','#3498DB'],
               linewidths=[1.5, 2.5, 1.5])
    ax.contourf(XX, YY, Z, levels=[-1, 0, 1],
                colors=['#FADBD8','#FDFEFE'], alpha=0.25)
    ax.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolors='k', s=60, zorder=3)
    ax.scatter(model.support_vectors_[:,0], model.support_vectors_[:,1],
               s=220, facecolors='none', edgecolors='gold', linewidths=2.5,
               label='Vectores de soporte', zorder=4)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)

fig, ax = plt.subplots(figsize=(7, 5))
plot_margin(ax, svm_hard, X_sep, y_sep, "SVM Hard Margin — Margen Máximo")
plt.tight_layout()
plt.savefig('svm_hard_margin.png', dpi=100, bbox_inches='tight')
plt.show()

w = svm_hard.coef_[0]
margin = 2 / np.linalg.norm(w)
print(f"Vectores de soporte: {len(svm_hard.support_vectors_)}")
print(f"Margen: {margin:.4f}")
print(f"Hiperplano: {w[0]:.3f}·x₁ + {w[1]:.3f}·x₂ + {svm_hard.intercept_[0]:.3f} = 0")

## Sección 2 — Soft Margin y el Parámetro C

En la práctica los datos tienen ruido y la separación perfecta puede no existir o sobreajustar.

**Soft Margin SVM** introduce variables de holgura ξᵢ ≥ 0 que permiten violaciones controladas.

| Valor de C | Comportamiento | Riesgo |
|-----------|----------------|--------|
| C → ∞ | Hard margin, cero violaciones | Sobreajuste |
| C grande | Pocos errores permitidos | Varianza alta |
| C pequeño | Muchos errores permitidos | Sesgo alto |

In [ ]:
# Efecto de C con datos solapados
X_ov, y_ov = make_blobs(n_samples=80, centers=2, cluster_std=1.6,
                         center_box=(-2, 2), random_state=RANDOM_STATE)
y_ov = 2 * y_ov - 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, C_val in zip(axes, [0.01, 1.0, 100.0]):
    svm_c = SVC(kernel='linear', C=C_val)
    svm_c.fit(X_ov, y_ov)
    acc = accuracy_score(y_ov, svm_c.predict(X_ov))
    margin_c = 2 / np.linalg.norm(svm_c.coef_[0])
    plot_margin(ax, svm_c, X_ov, y_ov,
                f"C = {C_val}\nSVs={len(svm_c.support_vectors_)}, "
                f"Margen={margin_c:.2f}, Acc={acc:.2f}")

plt.suptitle("Efecto de C en el Soft Margin SVM", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('svm_soft_margin.png', dpi=100, bbox_inches='tight')
plt.show()

## Sección 3 — El Truco del Kernel

Cuando los datos **no son linealmente separables** en el espacio original, podemos proyectarlos a un espacio de mayor dimensión donde sí lo sean.

El **truco del kernel** hace esto eficientemente sin calcular la proyección explícita.

| Kernel | Fórmula | Uso típico |
|--------|---------|-----------|
| Lineal | K(x,z) = x·z | Alta dimensión, texto |
| **RBF** | K(x,z) = exp(−γ‖x−z‖²) | **Uso general** |
| Polinomial | K(x,z) = (x·z + r)^d | Relaciones polinomiales |

In [ ]:
# Kernels en datos no-lineales
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
datasets_k = [
    (make_moons(n_samples=200, noise=0.15, random_state=RANDOM_STATE), "Moon Dataset"),
    (make_circles(n_samples=200, noise=0.1, factor=0.5, random_state=RANDOM_STATE), "Circles Dataset"),
]
kernels_cfg = [
    ('linear', {'C': 1.0}, 'Lineal'),
    ('rbf',    {'C': 1.0, 'gamma': 'scale'}, 'RBF (γ=scale)'),
    ('poly',   {'C': 1.0, 'degree': 3, 'gamma': 'scale'}, 'Polinomial (d=3)'),
]

for row, ((X_k, y_k), ds_name) in enumerate(datasets_k):
    for col, (kernel, params, kname) in enumerate(kernels_cfg):
        ax = axes[row, col]
        svm_k = SVC(kernel=kernel, **params)
        svm_k.fit(X_k, y_k)
        acc = accuracy_score(y_k, svm_k.predict(X_k))

        h = 0.02
        x_min, x_max = X_k[:,0].min()-0.3, X_k[:,0].max()+0.3
        y_min, y_max = X_k[:,1].min()-0.3, X_k[:,1].max()+0.3
        xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
        Z = svm_k.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, Z, cmap='RdBu', alpha=0.3)
        ax.scatter(X_k[:,0], X_k[:,1], c=y_k, cmap='RdBu', edgecolors='k', s=40)
        ax.scatter(svm_k.support_vectors_[:,0], svm_k.support_vectors_[:,1],
                   s=180, facecolors='none', edgecolors='gold', linewidths=2)
        ax.set_title(f"{ds_name} | {kname}\nAcc = {acc:.2f}", fontsize=10, fontweight='bold')
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.savefig('svm_kernels.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Kernel lineal no puede separar moon/circles — kernel RBF sí.")

## Sección 4 — Efecto de γ (gamma) en Kernel RBF

In [ ]:
X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=RANDOM_STATE)
gamma_values = [0.01, 0.1, 1.0, 10.0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, g in zip(axes, gamma_values):
    svm_g = SVC(kernel='rbf', C=1.0, gamma=g)
    svm_g.fit(X_moon, y_moon)
    h = 0.02
    x_min, x_max = X_moon[:,0].min()-0.3, X_moon[:,0].max()+0.3
    y_min, y_max = X_moon[:,1].min()-0.3, X_moon[:,1].max()+0.3
    xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
    Z = svm_g.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap='RdBu', alpha=0.3)
    ax.scatter(X_moon[:,0], X_moon[:,1], c=y_moon, cmap='RdBu', edgecolors='k', s=40)
    acc_tr = accuracy_score(y_moon, svm_g.predict(X_moon))
    ax.set_title(f"γ = {g}\nAcc(train) = {acc_tr:.2f}", fontsize=10, fontweight='bold')
    ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)

plt.suptitle("Efecto de γ en Kernel RBF (C=1.0)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('svm_gamma.png', dpi=100, bbox_inches='tight')
plt.show()
print("γ pequeño → frontera suave (underfitting) | γ grande → frontera irregular (overfitting)")

## Sección 5 — Caso Real: Dataset Wine + Grid Search

**Importante:** SVM es muy sensible a la escala — siempre normalizar con `StandardScaler`.

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
X_tr, X_te, y_tr, y_te = train_test_split(
    X_wine, y_wine, test_size=0.2, stratify=y_wine, random_state=RANDOM_STATE)

print(f"Wine dataset: {X_wine.shape[0]} muestras, "
      f"{X_wine.shape[1]} features, {len(wine.target_names)} clases")
print(f"Train: {len(X_tr)} | Test: {len(X_te)}")

In [ ]:
# Pipeline: StandardScaler + SVM
pipeline = Pipeline([('scaler', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                                 random_state=RANDOM_STATE))])
pipeline.fit(X_tr, y_tr)
y_pred = pipeline.predict(X_te)
print(f"Accuracy (sin tuning): {accuracy_score(y_te, y_pred):.4f}")
print()
print(classification_report(y_te, y_pred, target_names=wine.target_names))

In [ ]:
# Grid Search: C y gamma
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1],
    'svm__kernel': ['rbf', 'linear']
}
gs = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('svm', SVC(random_state=RANDOM_STATE))]),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
gs.fit(X_tr, y_tr)
print(f"Mejores parámetros: {gs.best_params_}")
print(f"CV score (train):   {gs.best_score_:.4f}")
print(f"Accuracy (test):    {accuracy_score(y_te, gs.best_estimator_.predict(X_te)):.4f}")

## Sección 6 — Vectores de Soporte

Los vectores de soporte son los puntos que **definen** el hiperplano. Eliminar cualquier otro punto no cambia el clasificador.

Este es el secreto de la **robustez** de SVM — solo depende de un subconjunto pequeño del dataset.

In [ ]:
# ¿Qué fracción del training son vectores de soporte?
svm_final = Pipeline([('scaler', StandardScaler()),
                       ('svm', SVC(kernel='rbf', C=10, gamma='scale',
                                   random_state=RANDOM_STATE))])
svm_final.fit(X_tr, y_tr)
svm_model = svm_final.named_steps['svm']

n_sv = svm_model.n_support_
total = len(X_tr)
print("="*55)
print("Análisis de Vectores de Soporte — Wine Dataset")
print("="*55)
print(f"Total puntos de entrenamiento: {total}")
for cls, n in zip(wine.target_names, n_sv):
    print(f"  SVs clase '{cls}': {n}")
print(f"  Total SVs: {n_sv.sum()} ({100*n_sv.sum()/total:.1f}% del training)")
print(f"\n✅ Solo el {100*n_sv.sum()/total:.1f}% de los puntos define el clasificador!")

## 📊 Resumen de la Clase

| Concepto | Descripción | Clave |
|----------|-------------|-------|
| Hard Margin | Sin tolerancia a errores | Separación perfecta |
| Soft Margin | Permite violaciones | Parámetro **C** |
| Kernel RBF | Proyección gaussiana | Parámetro **γ** |
| Vectores de soporte | Puntos que definen el margen | Son pocos |
| Normalización | Imprescindible en SVM | StandardScaler |

### Flujo de trabajo recomendado:
1. Normalizar features (StandardScaler).
2. Probar kernel RBF con GridSearchCV (C, γ).
3. Evaluar con cross-validation, reportar en test set.
4. Interpretar número de vectores de soporte.

---
*Conexión: en el Lab aplicarás esto al dataset `breast_cancer`; en el Assignment al dataset `wine`.*